# 92 · Ablation — false-positive building size

**Purpose:** Test whether the vector datasets' **false positives are disproportionately tiny buildings.**
The pipeline counts FP/FN but discards their sizes; this notebook re-runs *only the vector IoU matching*
(reusing `src`) and keeps every building's `area_m2` with its class (TP / FP / FN).

**Inputs:** `configs/validation_configs.yaml`, `aoi_tracker.csv`, reference + candidate vector files under `data/01_raw/`.

**Outputs:** `outputs/ablation_fp_size/fp_size_hist.csv` (10 m² histogram, re-aggregatable) + optional
`fp_size_buildings.parquet` (per-building rows) + ECDF / median figures.

**Run order:** standalone diagnostic — does **not** touch the pipeline, its configs, or its outputs.

**Last run:** _(fill in when you run it)_

In [ ]:
from google.colab import drive
import os, sys
from pathlib import Path
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/urban_validation')  # ← set to your project/data root

# Code from GitHub (for src), data from Drive.
import subprocess as _sp
_sp.run(['wget','-q','-O','/content/colab_bootstrap.py','https://raw.githubusercontent.com/GFDRR/urban_validation/fix/pipeline-audit/colab_bootstrap.py'], check=False)
sys.path.insert(0, '/content')
sys.modules.pop('colab_bootstrap', None); assert 'def setup' in open('/content/colab_bootstrap.py').read(), 'bootstrap download failed - check branch/URL'; from colab_bootstrap import setup as _setup
_setup(PROJECT_ROOT)   # clones repo -> sys.path; cwd stays on Drive

## Configuration — the only cell you normally edit

Flip `SAMPLE_MODE` off only after a test run. `MIN_AREA_M2` is the pipeline's 20 m² floor; drop it to
catch sub-20 m² micro-polygons (a sensitivity toggle, not the pipeline definition).

In [ ]:
# ── Scope / speed ────────────────────────────────────────────────────────────
SAMPLE_MODE   = True     # ← True: quick test on SAMPLE_CITIES; False: every suitable city
SAMPLE_CITIES = [        # balanced, memory-safe: 25 SpaceNet + 25 other (skips the RAM-heavy cities)
    'afg-bazarak', 'afg-charikar', 'afg-mahmud-e-raqi', 'afg-poruns',
    'ant-curacao', 'are-abudhabi', 'aus-melbourne', 'bgd-dhaka',
    'chn-lujiang', 'chn-shanghai', 'chn-zhuhai', 'egy-cairo',
    'gbr-birmingham', 'gha-dansoman', 'gha-kumasi', 'gha-nawuni',
    'grd-grenada', 'ind-mumbai', 'jam-saint-catherine', 'kor-sejong',
    'lby-almarj', 'lby-bayda', 'lby-benghazi', 'maf-saint-martin',
    'moz-djonasse', 'per-cusco', 'phl-catanduanes', 'phl-viga',
    'rou-bucharest', 'rus-astrakhan', 'sau-riyadh', 'sen-dakar',
    'sle-cockle-bay-1', 'sle-kolleh', 'tjk-artuch', 'tjk-tavishi-bolo',
    'ton-nukualofa', 'ton-sopu', 'tto-la-brea', 'uga-kampala-sn7',
    'uga-kanara', 'ukr-pulyny', 'usa-allentown', 'usa-atlanta',
    'usa-permianbasin', 'usa-portstlucie', 'usa-sandiego', 'yem-dhamar',
    'zaf-capetown', 'zmb-lusaka',
]

# Extreme candidate-count cities skipped even in FULL mode (OOM risk on any runtime).
CITY_EXCLUDE = ['ssd-juba', 'bgd-rohingya']

# ── Analysis knobs ───────────────────────────────────────────────────────────
MIN_AREA_M2      = 20.0   # pipeline floor; set lower (e.g. 0) to admit micro-polygons
BIN_M2           = 10     # histogram bin width (fine — re-aggregate later)
SAVE_PER_BUILDING = SAMPLE_MODE   # per-building parquet (exact percentiles); heavy on a full run
REUSE_EXISTING    = True           # if fp_size_hist.csv already exists, skip the (slow) re-match

# Unified tracker with the reference_source column (drives τ). Override if yours differs.
AOI_TRACKER = 'data/02_interim/aoi_tracker.csv'

# City-adaptive IoU threshold — same rule as the vector pipeline.
TAU_SPACENET7 = 0.25
TAU_DEFAULT   = 0.50

VEC_DATASETS = ['overture', 'gba', 'globfp']
OUT_DIR = PROJECT_ROOT / 'outputs' / 'ablation_fp_size'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUT_DIR)

## Setup — reuse the pipeline's own functions

In [ ]:
import time, warnings, gc
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import geopandas as gpd
import yaml
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from src.utils.buildings import load_buildings
from src.utils.tiling import make_tiles, subset_by_tile
from src.utils.geometry import get_projected_crs
from src.metrics.vector.matching import match_buildings_iou
from src.utils.aoi_inventory import load_validation_datasets
from src.plots.style import DATASET_COLORS, DATASET_LABELS

def save_fig(fig, name):
    """Vector PDF (Overleaf) + 300 dpi PNG (Docs/PPT)."""
    for ext in ('pdf', 'png'):
        fig.savefig(f'{OUT_DIR}/{name}.{ext}', dpi=300, bbox_inches='tight')

# Load config, point it at PROJECT_ROOT + the unified tracker, list the cities.
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'validation_configs.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir']    = str(PROJECT_ROOT)
cfg['aoi_tracker'] = AOI_TRACKER
DATA_DIR   = PROJECT_ROOT / cfg['data_dir']
TILE_SIZE  = float(cfg['vector']['preprocessing']['tile_size_m'])
TAU_BUFFER = float(cfg['vector']['preprocessing'].get('tau_buffer_m', 0.0))

datasets = load_validation_datasets(cfg, DATA_DIR)
by_id = {d['id']: d for d in datasets}
cities = [c for c in SAMPLE_CITIES if c in by_id] if SAMPLE_MODE else [d['id'] for d in datasets]
cities = [c for c in cities if c not in CITY_EXCLUDE]
missing = [c for c in SAMPLE_CITIES if c not in by_id] if SAMPLE_MODE else []
print(f'{len(datasets)} suitable cities in tracker; running {len(cities)} '
      f'({"SAMPLE" if SAMPLE_MODE else "FULL"}).')
if missing:
    print('  not found in tracker (skipped):', missing)
if not SAMPLE_MODE and SAVE_PER_BUILDING:
    print('  WARNING: SAVE_PER_BUILDING on a full run writes a very large parquet.')

## Re-match & classify

For each city × dataset × tile: match at the city's τ, then label every building — candidate matched →
**TP**, unmatched → **FP**; reference matched → **TP**, unmatched → **FN** — and bin its `area_m2` into
10 m² buckets. Edge buildings follow the pipeline's per-tile *intersects* rule (a boundary building can
appear in two tiles), so FP/FN here match the pipeline's counts exactly.

In [ ]:
hist_path = OUT_DIR / 'fp_size_hist.csv'
if REUSE_EXISTING and hist_path.exists():
    print(f'✓ Reusing existing {hist_path.name} — skipping the re-match. '
          'Set REUSE_EXISTING=False (or delete the CSV) to recompute.')
else:
    hist = defaultdict(Counter)     # (city, dataset, side, klass) -> Counter{bin_lo: count}
    raw_rows = []                   # per-building rows (only if SAVE_PER_BUILDING)

    def _accumulate(city, ds, side, klass, areas, tile_id):
        if len(areas) == 0:
            return
        areas = np.asarray(areas, dtype='float64')
        bins = (np.floor(areas / BIN_M2).astype(int) * BIN_M2)
        u, cnt = np.unique(bins, return_counts=True)
        hist[(city, ds, side, klass)].update(dict(zip(u.tolist(), cnt.tolist())))
        if SAVE_PER_BUILDING:
            raw_rows.extend({'city': city, 'dataset': ds, 'tile_id': int(tile_id),
                             'side': side, 'class': klass, 'area_m2': float(a)} for a in areas)

    t0 = time.time()
    for n, city in enumerate(cities, 1):
        d = by_id[city]
        ref_source = str(d.get('reference_source', 'other')).lower().strip()
        tau = TAU_SPACENET7 if ref_source in ('spacenet', 'spacenet7') else TAU_DEFAULT

        crs = get_projected_crs(d['aoi'])
        tiles = make_tiles(d['aoi'].to_crs(crs), TILE_SIZE)

        ref_parts = []
        for p in d.get('ref_paths', []):
            if Path(p).exists():
                try:
                    ref_parts.append(load_buildings(p, crs_work=crs, min_area_m2=MIN_AREA_M2, fix_invalid_geoms=True))
                except Exception as e:
                    print(f'  [ref] {city}: {Path(p).name}: {e}')
        if not ref_parts:
            print(f'  [skip] {city}: no reference buildings'); continue
        ref_all = gpd.GeoDataFrame(pd.concat(ref_parts, ignore_index=True), crs=ref_parts[0].crs)
        ref_sindex = ref_all.sindex

        slug = city.lower().replace('-', '_')
        for ds in VEC_DATASETS:
            cf = sorted((DATA_DIR / city / 'vector').glob(f'{slug}_{ds}*.parquet'))
            if not cf:
                continue
            try:
                cand_all = load_buildings(cf[0], crs_work=crs, min_area_m2=MIN_AREA_M2, fix_invalid_geoms=True)
            except Exception as e:
                print(f'  [cand] {city}/{ds}: {e}'); continue
            cand_sindex = cand_all.sindex

            for t in tiles.itertuples():
                geom = t.geometry
                rt = subset_by_tile(ref_all, ref_sindex, geom)
                ct = subset_by_tile(cand_all, cand_sindex, geom)
                m, ref_un, cand_un = match_buildings_iou(rt, ct, tau, tau_buffer_m=TAU_BUFFER)
                tid = int(t.tile_id)
                if not m.empty:
                    _accumulate(city, ds, 'candidate', 'TP', m['area_cand'].values, tid)
                    _accumulate(city, ds, 'reference', 'TP', m['area_ref'].values,  tid)
                if cand_un:
                    _accumulate(city, ds, 'candidate', 'FP', ct.loc[list(cand_un), 'area_m2'].values, tid)
                if ref_un:
                    _accumulate(city, ds, 'reference', 'FN', rt.loc[list(ref_un), 'area_m2'].values, tid)
            del cand_all, cand_sindex; gc.collect()
        del ref_all, ref_sindex; gc.collect()
        print(f'  [{n}/{len(cities)}] {city}  (τ={tau}, {ref_source})  {time.time()-t0:.0f}s')

    # ── flatten histogram -> tidy CSV ────────────────────────────────────────────
    rows = [{'city': k[0], 'dataset': k[1], 'side': k[2], 'class': k[3],
             'size_lo_m2': b, 'count': c}
            for k, ctr in hist.items() for b, c in ctr.items()]
    hist_df = pd.DataFrame(rows).sort_values(['dataset', 'side', 'class', 'city', 'size_lo_m2'])
    hist_path = OUT_DIR / 'fp_size_hist.csv'
    hist_df.to_csv(hist_path, index=False)
    print(f'\nSaved histogram: {hist_path}  ({len(hist_df):,} rows, bin={BIN_M2} m²)')

    if SAVE_PER_BUILDING and raw_rows:
        raw_df = pd.DataFrame(raw_rows)
        raw_path = OUT_DIR / 'fp_size_buildings.parquet'
        raw_df.to_parquet(raw_path, index=False)
        print(f'Saved per-building rows: {raw_path}  ({len(raw_df):,} buildings)')

## Analysis — are the false positives tiny?

In [ ]:
# Reload the histogram (so this cell also works stand-alone from a prior run).
hist_df = pd.read_csv(OUT_DIR / 'fp_size_hist.csv')

# Size classes — refined at the small end; re-bin freely from the 10 m² histogram.
SIZE_BREAKS = [MIN_AREA_M2, 40, 60, 80, 120, 180, np.inf]
SIZE_LABELS = [f'{int(MIN_AREA_M2)}–40', '40–60', '60–80', '80–120', '120–180', '>180']
hist_df['cls'] = pd.cut(hist_df['size_lo_m2'], bins=SIZE_BREAKS, labels=SIZE_LABELS, right=False)

def _classes(dataset, side):
    g = (hist_df[(hist_df['dataset'] == dataset) & (hist_df['side'] == side)]
         .groupby(['cls', 'class'], observed=True)['count'].sum().unstack('class')
         .reindex(SIZE_LABELS).fillna(0))
    return g
def _col(df, name):
    return df[name] if name in df.columns else pd.Series(0.0, index=df.index)
def _rate(df, num, den1, den2):
    d = _col(df, den1) + _col(df, den2)
    return (100 * _col(df, num) / d.replace(0, np.nan))

# ── (1) size summary table (median/p25/p75 per side & class) ──────────────────
def _pool(dataset, side, klass):
    g = (hist_df[(hist_df['dataset'] == dataset) & (hist_df['side'] == side) & (hist_df['class'] == klass)]
         .groupby('size_lo_m2')['count'].sum().sort_index())
    return g.index.values.astype(float), g.values.astype(float)
def _q(bl, c, q):
    if c.sum() == 0: return np.nan
    cum = np.cumsum(c); tgt = q * cum[-1]; i = min(int(np.searchsorted(cum, tgt)), len(bl) - 1)
    prev = cum[i-1] if i > 0 else 0.0
    return bl[i] + ((tgt - prev)/c[i] if c[i] > 0 else 0.0) * BIN_M2
summary = []
for ds in VEC_DATASETS:
    for side, klass in (('candidate','TP'), ('candidate','FP'), ('reference','FN')):
        bl, c = _pool(ds, side, klass)
        summary.append({'dataset': DATASET_LABELS.get(ds, ds), 'side': side, 'class': klass,
                        'n': int(c.sum()), 'median_m2': round(_q(bl, c, .50), 1),
                        'p25_m2': round(_q(bl, c, .25), 1), 'p75_m2': round(_q(bl, c, .75), 1)})
summary_df = pd.DataFrame(summary)
print('Building size by class (pooled across cities):')
print(summary_df.to_string(index=False))
summary_df.to_csv(OUT_DIR / 'fp_size_summary.csv', index=False)

# ── (2) headline: false-discovery rate + miss rate vs building size ───────────
x = np.arange(len(SIZE_LABELS))
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ds in VEC_DATASETS:
    col = DATASET_COLORS.get(ds, '#888888'); lab = DATASET_LABELS.get(ds, ds)
    axL.plot(x, _rate(_classes(ds, 'candidate'), 'FP', 'TP', 'FP').values, '-o', color=col, lw=2, ms=5, label=lab)
    axR.plot(x, _rate(_classes(ds, 'reference'), 'FN', 'TP', 'FN').values, '-o', color=col, lw=2, ms=5, label=lab)
for ax, ttl, sub, xl in [
    (axL, 'False-discovery rate  (candidate side)', 'FP / (TP + FP) — spurious detections', 'Candidate building size (m²)'),
    (axR, 'Miss rate  (reference side)',            'FN / (TP + FN) — missed real buildings', 'Reference building size (m²)')]:
    ax.axhline(50, color='#cccccc', lw=0.8, zorder=0)
    ax.set_xticks(x); ax.set_xticklabels(SIZE_LABELS); ax.set_xlabel(xl)
    ax.set_ylim(0, 100); ax.grid(alpha=0.3); ax.set_title(ttl, fontweight='bold', fontsize=12)
    ax.text(0.5, 1.002, sub, transform=ax.transAxes, ha='center', va='bottom', fontsize=9, color='#555')
axL.set_ylabel('% of buildings'); axL.legend(fontsize=9, loc='upper right')
fig.suptitle('Why small buildings score low: spurious small candidates (left) + missed small references (right)',
             fontsize=13, y=1.05)
fig.tight_layout()
save_fig(fig, 'AB1_precision_recall_by_size')
plt.show()

# ── (3) companion: candidate building COUNTS by size — TP vs FP ───────────────
fig2, axes = plt.subplots(1, len(VEC_DATASETS), figsize=(4.6*len(VEC_DATASETS), 4.2), sharey=True)
axes = np.atleast_1d(axes); w = 0.4
for ax, ds in zip(axes, VEC_DATASETS):
    cc = _classes(ds, 'candidate')
    ax.bar(x - w/2, _col(cc, 'TP').values/1e3, w, color=DATASET_COLORS.get(ds, '#888888'), alpha=0.6, label='TP (matched)')
    ax.bar(x + w/2, _col(cc, 'FP').values/1e3, w, color='#8A8A96', label='FP (spurious)')
    ax.set_xticks(x); ax.set_xticklabels(SIZE_LABELS, rotation=45, ha='right')
    ax.set_title(DATASET_LABELS.get(ds, ds), color=DATASET_COLORS.get(ds, '#888888'), fontweight='bold')
    ax.set_xlabel('Building size (m²)'); ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel('Candidate buildings (thousands)')
fig2.suptitle('Candidate building instances by size — true vs false positives', fontsize=13, y=1.02)
fig2.tight_layout()
save_fig(fig2, 'AB2_tp_vs_fp_counts')
plt.show()
print('\nLeft: smaller candidates are more likely spurious.  Right: small real buildings are missed more '
      '(severe for GlobFP/GBA).  Overture = over-detects small; GlobFP = under-detects small; GBA = both.')

**Interpretation — two size-dependent failure modes, accounting for both FP and FN.**

- **Left / false-discovery rate** (`FP/(TP+FP)` by candidate size): small candidate footprints are disproportionately spurious — up to ~80% of 20–40 m² candidates are false positives.
- **Right / miss rate** (`FN/(TP+FN)` by reference size): small *real* buildings are disproportionately **missed** — severe for GlobFP (~75% at 20–40 m²) and GBA, negligible for Overture.

Together they decompose the report's "F1 drops for small buildings" into precision vs recall. Dataset personalities: Overture over-detects small (high FP, low miss); GlobFP under-detects small (low FP at large sizes, high miss); GBA does both. Everything derives from the 10 m² histogram, so re-bin or add cities without re-matching. The pipeline's `MIN_AREA_M2 = 20` floor applies before matching; lower it to probe sub-20 m² micro-polygons.